In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings 
warnings.filterwarnings("ignore")

In [1]:
from DFTStructureGenerator import B_N_Cl, xtb_process, mol_manipulation
import glob, os, shutil, itertools, copy
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from tqdm import tqdm
import pandas as pd

In [4]:
# DFT Method In Need
#p opt=(maxcycles=150) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm
OPT_METHOD = "opt freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
SPE_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm"
SPE_WFN_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm output=wfn"
OM_METHOD = "opt=(modredundant,maxcycles=100) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
TS_METHOD = "opt=(calcfc,ts,noeigen,maxcycles=100) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
SPE_SPECIAL = "b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"

In [ ]:
# root_file = "G:/work/B_Cl_Nu/Data"                    # All Reactants save in this dir
root_file = "G:/work/B_Cl_Nu/Data" 
if not os.path.isdir(root_file):
    os.mkdir(root_file)
mol_xtb_file = os.path.join(root_file, 'Mol_xtb')       
if not os.path.isdir(mol_xtb_file):
    os.mkdir(mol_xtb_file)         
mol_dir = os.path.join(root_file, 'Mols')              
dft_dir = os.path.join(root_file, 'GS_OPT')             
spe_dir = os.path.join(root_file, 'GS_SPE')                

文件目录嵌套关系
- Data              # 所有数据储存的地址，即root_file
  - GS_OPT          # 基态优化的DFT文件夹
    - B_N_r         # B自由基和配体的复合物-反应物自由基的优化
    - B_N_r_improve # 脚本默认生成的Gaussian改错文件地址
    - dustbin       # 计算失败的log
  - Mols            # Mol文件夹
  - Mol_xtb         # Xtb计算的输出地址


# 1 数据整理

In [ ]:
react_csv = pd.read_excel('Data/All_Data/Data_Structure_Update_Sum.xlsx')
NHC_smiles = react_csv["Ar"].dropna().to_numpy()
for each in NHC_smiles:
    if Chem.MolFromSmiles(each) == None:
        print(each)

In [ ]:
duplicate_N_id, duplicate_Cl_id = B_N_Cl.generate_combinations(
    reactant_file='Data/All_Data/Data_Structure_Update_250619.xlsx',
    Bresult_file='Data/All_Data/reactants_B.csv',
    Nresult_file='Data/All_Data/reactants_N.csv',
    Clresult_file='Data/All_Data/reactants_Cl.csv'
)

In [5]:
duplicate_N_id = [9, 43, 285, 310, 314, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 372, 375, 376]
duplicate_Cl_id = [624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636,
       637, 638, 639, 640, 642, 644, 645, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664,
       665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677,
       678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 690,
       691, 692, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703,
       704, 705, 706, 707, 708, 709, 710, 711, 713, 714, 716, 717, 718, 719, 720, 721, 722]

## 1 产生B\N单体优化结构，XTB找构象

In [ ]:
B_N_Cl.B_N_Single_Xtb(root_file=root_file,
    Bresult_file='Data/All_Data/reactants_B.csv', 
    Nresult_file='Data/All_Data/reactants_N.csv', 
    mol_xtb_name = "Mol_xtb_2")


In [6]:
B_N_Cl.B_N_Cl_reactant_product_Xtb(root_file=root_file,
    Bresult_file='Data/All_Data/reactants_B.csv', 
    Nresult_file='Data/All_Data/reactants_N.csv', 
    Clresult_file='Data/All_Data/reactants_Cl.csv',
    duplicate_N_id=duplicate_N_id,
    duplicate_Cl_id=duplicate_Cl_id,
    mol_xtb_name = 'Mol_xtb_2', 
    mol_name = "Mols2")
    

In [7]:
xtb_process.shift_to_parra(r"G:\work\B_Cl_Nu\Data\Mol_xtb_2\Cl_r", 0, 0)
xtb_process.shift_to_parra(r"G:\work\B_Cl_Nu\Data\Mol_xtb_2\Cl_p", 0, 1)
xtb_process.shift_to_parra(r"G:\work\B_Cl_Nu\Data\Mol_xtb_2\Cl_p_d", 0, 1)

# 2 DFT结构搭建

In [8]:
all_strs = ['Cl_r', 'B_N_r', 'Cl_p', 'Cl_p_d', 'B_N_p', 'B_N_p_d', 'B_N_r_d', "N_single"]
all_spins = [1,2,2,2,1,1,2,1]
all_strs = ['Cl_r','Cl_p_d', "Cl_p"]
all_spins = [1, 2, 2]
for name, spin in zip(all_strs, all_spins):
    mol_xtb_file = os.path.join(root_file, 'Mol_xtb_2')
    root_dir = mol_xtb_file + "/" + name
    mol_dir = os.path.join(root_file, 'Mols2')
    dft_dir = root_file + "/GS_OPT2"
    if not os.path.isdir(dft_dir):
        os.mkdir(dft_dir)

    dft_dir_ = root_file + "/GS_OPT2" + "/" + name
    if not os.path.isdir(dft_dir_):
        os.mkdir(dft_dir_)

    B_N_Cl.smiles_DFT_calc(root_dir, mol_dir, dft_dir_, method = OPT_METHOD, conf_limit = 10, SpinMultiplicity=spin) 

mol_str:Cl_00721_r didn't find crest_best!
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00004_p_0000.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00004_p_0001.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00004_p_0002.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00004_p_0003.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00004_p_0004.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00004_p_0005.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00007_p_0000.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00007_p_0001.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_p_d/Cl_00721_Claid_00007_p_0002.gjf 's SpinMultiplicity != 1, check it
G:/work/B_Cl_Nu/Data/GS_OPT2/C

In [13]:
# 错误排查
mol_manipulation.error_improve(dft_dir, mol_dir, 'Cl_r', 'dust_bin', improve_dir='Cl_p_d_imp')

G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_r\Cl_00722_r_0002.log didn't run successful
G:/work/B_Cl_Nu/Data/GS_OPT2/Cl_r\Cl_00722_r_0002.log. Error Reason is link 9999 or unfinished.


In [16]:
B_N_Cl.SPE_DFT_calc(root_file, 'GS_OPT2/Cl_p_d', 'GS_SPE2/Cl_p_d', "Mols2", method=SPE_METHOD)

In [24]:
spe_dir = os.path.join(root_file, 'GS_SPE2')

In [19]:
mol_manipulation.error_improve(spe_dir, mol_dir, 'Cl_p_d', 'dust_bin', improve_dir='Cl_p_d_imp')

In [ ]:
mol_manipulation.error_improve(spe_dir, mol_dir, 'B_N_p_new', 'dust_bin', improve_dir='B_N_p_new_imp')

In [22]:
B_N_Cl.SPE_DFT_calc_wfn(root_file, 'GS_OPT2/Cl_p_d', 'GS_SPE_wfn2/Cl_p_d', "Mols2", method=SPE_WFN_METHOD)

100%|██████████| 12/12 [00:00<00:00, 12.49it/s]


In [ ]:
B_N_Cl.SPE_DFT_calc_wfn(root_file, 'GS_OPT/N_single', 'GS_SPE_wfn/N_single', method=SPE_WFN_METHOD)

In [ ]:
B_N_Cl.SPE_DFT_calc_wfn(root_file, 'GS_OPT/B_N_p_d', 'GS_SPE_wfn/B_N_p_d', method=SPE_WFN_METHOD)

100%|██████████| 49203/49203 [08:14<00:00, 99.45it/s] 


In [25]:
# 反应物计算完成整理
B_N_Cl.collection_dft_single(
    Bresult_path = 'Data/All_Data/reactants_B.csv', 
    Nresult_path = 'Data/All_Data/reactants_N.csv',
    Clresult_path = 'Data/All_Data/reactants_Cl.csv', 
    mol_dir=mol_dir, dft_dir=dft_dir, spe_dir=spe_dir, 
    duplicate_Cl_id = duplicate_Cl_id)


55it [00:00, 58.61it/s]
415it [00:16, 25.45it/s]
386it [00:10, 35.44it/s]
386it [00:14, 26.97it/s]


In [ ]:
# 复合物整理
B_N_Cl.collection_dft_couple(
    Bresult_path = 'Data/All_Data/reactants_B.csv',
    Nresult_path = 'Data/All_Data/reactants_N.csv',
    B_N_result_path = 'Data/All_Data/reactants_B_N.csv',
    type_ = 'r',
    mol_dir=mol_dir, dft_dir=dft_dir, spe_dir=spe_dir,
    duplicate_N_id = duplicate_N_id
)

In [48]:
# 复合物整理2
B_N_Cl.collection_dft_couple(
    Bresult_path = 'Data/All_Data/reactants_B.csv',
    Nresult_path = 'Data/All_Data/reactants_N.csv',
    B_N_result_path = 'Data/All_Data/reactants_B_N.csv',
    type_ = 'p',
    mol_dir=mol_dir, dft_dir=dft_dir, spe_dir=spe_dir,
    duplicate_N_id = duplicate_N_id
)

55it [1:46:54, 116.64s/it]


In [ ]:
data_csv = pd.read_csv('Data/All_Data/reactants_B_N.csv')
data_csv = data_csv.loc[[id for id, row in data_csv.iterrows() if not np.isnan(row['deltaG_comb(kcal)']) and not np.isnan(row['deltaG_react'])]]
data_csv.to_csv('Data/All_Data/reactants_B_N.csv', index=False)
data_csv = pd.read_csv('Data/All_Data/reactants_Cl.csv')
data_csv = data_csv.loc[[id for id, row in data_csv.iterrows() if not np.isnan(row['G_energy_p'])]]
data_csv.to_csv('Data/All_Data/reactants_Cl.csv', index=False)

In [ ]:
data_csv = pd.read_csv('Data/All_Data/reactants_Cl.csv')
data_csv = data_csv.loc[[id for id, row in data_csv.iterrows() if not np.isnan(row['G_energy_p'])]]
data_csv['deltaG_react'] = data_csv['G_energy_p'] - data_csv['G_energy_r']
data_csv.to_csv('Data/All_Data/reactants_Cl.csv', index=False)